In [4]:
from transformers import AutoModel, AutoTokenizer
import torch
from datasets import load_dataset

In [5]:

ds = load_dataset("CATMuS/medieval")

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 152816
    })
    validation: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 19402
    })
    test: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 22590
    })
})

In [7]:
import os
import json
from tqdm import tqdm


def save_dataset_to_disk(dataset_dict, output_root="local_dataset"):

    
    # Create root directory
    os.makedirs(output_root, exist_ok=True)
    
    for split_name, dataset in dataset_dict.items():
        print(f"Processing split: {split_name}...")
        
        # Create split directory (e.g., local_dataset/train)
        split_dir = os.path.join(output_root, split_name)
        os.makedirs(split_dir, exist_ok=True)
        
        metadata_path = os.path.join(split_dir, "metadata.jsonl")
        
        # Open metadata file for writing
        with open(metadata_path, "w", encoding="utf-8") as f:
            # Use tqdm for a progress bar since the dataset is large
            for i, example in tqdm(enumerate(dataset), total=len(dataset)):
                
                # 1. Handle the Image
                image = example["im"]
                
                # Create a unique filename. 
                # We use the index 'i' to guarantee uniqueness.
                # If you have a unique ID in the data, you could use that too.
                image_filename = f"image_{i}.png"
                image_path = os.path.join(split_dir, image_filename)
                
                # Save image
                # Convert to RGB to avoid errors with CMYK/RGBA formats
                if image.mode != "RGB":
                    image = image.convert("RGB")
                image.save(image_path)
                
                # 2. Handle the Metadata (Text + others)
                # We create a dictionary that links the filename to the text
                metadata_entry = {
                    "file_name": image_filename,
                    "text": example["text"]
                }
                
                # Optional: Add other useful fields if you want them for filtering later
                # (e.g., you might want to filter by century or language later)
                extra_fields = ['century', 'language', 'script_type']
                for field in extra_fields:
                    if field in example:
                        metadata_entry[field] = example[field]
                
                # Write to jsonl
                f.write(json.dumps(metadata_entry) + "\n")
                
    print(f"\nSuccess! Dataset saved to '{output_root}'")

# --- Usage ---
# Assuming 'ds' is your loaded DatasetDict
# save_dataset_to_disk(ds, output_root="catmus_images")

save_dataset_to_disk(ds)

Processing split: train...


  7%|▋         | 10738/152816 [06:12<1:22:12, 28.81it/s]


KeyboardInterrupt: 

In [5]:
model_name = 'deepseek-ai/DeepSeek-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModel.from_pretrained("deepseek-ai/DeepSeek-OCR", trust_remote_code=True, torch_dtype="auto")
model = model.eval().cuda().to(torch.bfloat16)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# prompt = "<image>\nFree OCR. "
prompt = "<image>\nFree OCR. "
image_file = 'local_dataset/train/image_0.png'
output_path = '../test'
# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=image_file,
    output_path = output_path,
    base_size = 1024,
    image_size = 640,
    crop_mode=True,
    save_results = True,
    test_compress = False)

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [17]:
inputs = processor(images=ds["train"][0]["im"], return_tensors="pt")

ValueError: You need to specify either `text` or `text_target`.

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 152816
    })
    validation: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 19402
    })
    test: Dataset({
        features: ['text', 'im', 'language', 'century', 'region', 'script_type', 'shelfmark', 'verse', 'genre', 'project', 'line_type', 'gen_split'],
        num_rows: 22590
    })
})

In [7]:
ds["train"][0]

{'text': 'q̃  y paresçio. ffecha ⁊ signada por man de domĩgo ffeĩrs escrͥuan publico de Burgos del rregistͦ de pͦ m̃z escͥuan. ffizierõ abenẽçia en esta maña q̃los dich̃s Sanch̃ ꝑez ⁊',
 'im': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=4772x170>,
 'language': 'Castilian',
 'century': 13,
 'region': 'MainZone',
 'script_type': 'Semihybrida',
 'shelfmark': 'Paris, BnF, esp. 480',
 'verse': 'prose',
 'genre': 'Documents of practice',
 'project': 'HTRomance',
 'line_type': 'DefaultLine',
 'gen_split': 'test'}

In [ ]:
# prompt = "<image>\nFree OCR. "
prompt = "<image>\nFree OCR. "
image_file = ds["train"][0]["im"]
output_path = '../test'
# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=image_file,
    output_path = output_path,
    base_size = 1024,
    image_size = 640,
    crop_mode=True,
    save_results = False,
    test_compress = False)

error: [Errno 2] No such file or directory: '<PIL.PngImagePlugin.PngImageFile image mode=RGB size=4772x170 at 0x74C0A43BF620>'


AttributeError: 'NoneType' object has no attribute 'convert'

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = '0'
model_name = 'deepseek-ai/DeepSeek-OCR'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, _attn_implementation='flash_attention_2', trust_remote_code=True, use_safetensors=True)
model = model.eval().cuda().to(torch.bfloat16)

# prompt = "<image>\nFree OCR. "
prompt = "<image>\n<|grounding|>Convert the document to markdown. "
image_file = '5bd4326803572048274d287c.jpg'
output_path = 'test'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)


/home/diego/Documents/clases/procesamiento lenguaje natural/proyecto/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


ValueError: DeepseekOCRForCausalLM does not support an attention implementation through torch.nn.functional.scaled_dot_product_attention yet. Please request the support for this architecture: https://github.com/huggingface/transformers/issues/28005. If you believe this error is a bug, please open an issue in Transformers GitHub repository and load your model with the argument `attn_implementation="eager"` meanwhile. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="eager")`

In [3]:
# Load model directly
from transformers import AutoModel, AutoTokenizer
import torch

model_name = 'deepseek-ai/DeepSeek-OCR'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModel.from_pretrained("deepseek-ai/DeepSeek-OCR", trust_remote_code=True, torch_dtype="auto")
model = model.eval().cuda().to(torch.bfloat16)

# prompt = "<image>\nFree OCR. "
prompt = "<image>\n<|grounding|>Convert the document to markdown. "
image_file = '5bd4326803572048274d287c.jpg'
output_path = 'test'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/diego/Documents/clases/procesamiento lenguaje natural/proyecto/venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain rel

BASE:  torch.Size([1, 256, 1280])
NO PATCHES
<|ref|>image<|/ref|><|det|>[[0, 0, 999, 999]]<|/det|>
image size:  (640, 360)
valid image tokens:  144
output texts tokens (valid):  18
compression ratio:  0.12
===============save results:===============


image: 100%|██████████| 1/1 [00:00<00:00, 17119.61it/s]
other: 0it [00:00, ?it/s]
